# Sentiment detection from text task

## IMDB Dataset

This is a dataset of 25,000 movie reviews from IMDB, labeled by sentiment (positive/negative).

The reviews have been preprocessed, and each review is encoded as a list of word indices (integers).

For convenience, words are indexed by overall frequency in the dataset, so that for instance the integer "3" encodes the third most frequent word in the data.

This allows for quick filtering operations, such as: "only consider the top 10,000 most common words, but exclude the top 20 most common words".

By convention, "0" does not stand for a specific word, but is used to encode an item token.

In [ ]:
from keras.datasets import imdb
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

## Loading the data
Again, we use a ready-made framework function to load the data.

In [ ]:
vocabulary_size = 5000
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=vocabulary_size)

Let's display the first training record.

The input data is encoded as words by index.

In [ ]:
print (X_train[0])

Let's look at how the review looks assembled from words instead of numbers.

First we need to download the vocabulary.

In [ ]:
word_idx = imdb.get_word_index()

Originally the index number is not the key for the value.

So we need to convert it so the index is the key and the word is the value.

In [ ]:
word_idx = {i: word for word, i in word_idx.items()}

Displaying the first review in text form.

In [ ]:
print([word_idx[i] for i in X_train[0]])

The first review has 218 words.

In [ ]:
len(X_train[0])

Let's find out how long the reviews are.

In [ ]:
print("Maximum length of a review: ", len(max((X_train + X_test), key=len)))
print("Minimum length of a review: ", len(min((X_train + X_test), key=len)))

Now that we know what the input data looks like, let's look at the output.

A review can be negative (0) or positive (1).

In [ ]:
print(np.unique(y_train))

In [ ]:
class_names = ["negative", "positive"]

# Data preparation
The tensorflow library has functions for working with sequences.

In [ ]:
from tensorflow.keras.preprocessing import sequence

We take the first 400 words from each review. If a review is not long enough, we pad it with an empty word, i.e. the number 0.

In [ ]:
max_words = 400
 
X_train = sequence.pad_sequences(X_train, maxlen=max_words)
X_test = sequence.pad_sequences(X_test, maxlen=max_words)
 
X_valid, y_valid = X_train[:64], y_train[:64]
X_train, y_train = X_train[64:], y_train[64:]

In [ ]:
print (X_train.shape)
print (X_test.shape)
print (X_valid.shape)

Let's check the length of the first review, which originally had 218 characters.

In [ ]:
print (len(X_train[0]))

Let's look at the first review.

In [ ]:
X_train[0]

The output data takes the form of the number 0 or 1.

In [ ]:
y_train[0]

Since we are building classification networks, it is a good idea to convert the output value into a vector of probabilities.

In [ ]:
from keras.utils import to_categorical 
y_train = to_categorical(y_train, num_classes=2)
y_test = to_categorical(y_test, num_classes=2)
y_valid = to_categorical(y_valid, num_classes=2)

In [ ]:
print (y_train[0])

In [ ]:
print (y_train.shape)
print (y_test.shape)
print (y_valid.shape)

# Simple RNN model
Again, we choose SimpleRNN for the neural network.

In [ ]:
from keras.layers import SimpleRNN, Dense, Embedding
from keras.models import Sequential

We create a sequential model

In [ ]:
RNN_model = Sequential(name="Simple_RNN")

The first layer will be Embedding, which is used to map discrete values (e.g. numeric word IDs) into dense vectors (embeddings).

It is typically used when working with text. We have a vocabulary of size vocabulary_size, and each word is represented by a number (an index in the vocabulary).

Embedding converts this number into a fixed-length vector embd_len.

This way, instead of one-hot encoding, words are represented by a more compact, meaningful vector.

We need to set the embedding size. In our case we set it to 32.

In [ ]:
embd_len = 32
RNN_model.add(Embedding(vocabulary_size, embd_len))

Then comes the SimpleRNN network.

In [ ]:
RNN_model.add(SimpleRNN(128,
                        activation='tanh',
                        return_sequences=False))

Last is the output Dense layer, which returns the positive/negative categories.

In [ ]:
RNN_model.add(Dense(2, activation='softmax'))

Displaying the neural network structure.

In [ ]:
RNN_model.summary()

This is a classification model with two classes, so we use the binary_crossentropy loss function.

In [ ]:
RNN_model.compile(
    loss="categorical_crossentropy",
    optimizer='adam',
    metrics=['accuracy']
)

### Training the model
Training the model takes a long time, so we only use 15 epochs.

In [ ]:
rnn_history = RNN_model.fit(X_train, y_train,
                        batch_size=64,
                        epochs=15,
                        verbose=1,
                        validation_data=(X_valid, y_valid))

We save the trained model.

In [ ]:
RNN_model.save('rnn_simple.keras')

### Training history

In [ ]:
fig1 = plt.figure()
plt.plot(rnn_history.history['loss'], label='Train Loss')
plt.plot(rnn_history.history['accuracy'], label='Train Accuracy')
plt.plot(rnn_history.history['val_loss'], label='Validation Loss')
plt.plot(rnn_history.history['val_accuracy'], label='Validation Accuracy')


plt.legend(loc="best")
plt.title('Loss, accuracy')
plt.ylabel('Loss, accuracy')
plt.xlabel('Number of epochs')
plt.show()   

### Model verification

In [ ]:
scores = RNN_model.evaluate(X_test, y_test)
print (f"Loss function: {scores[0]}")
print (f"Accuracy: {scores[1]}")

Predicting the test data

In [ ]:
y_pred = RNN_model.predict(X_test)

Evaluation results for the first review

In [ ]:
print (f"Prediction: {y_pred[0]}")
print (f"Reality: {y_test[0]}")

Predictions and reality

In [ ]:
y_pred_best_answer = np.argmax(y_pred, axis=-1)
y_test_best_answer=np.argmax(y_test, axis=-1)
print (f"Predictions: {y_pred_best_answer}")
print (f"Reality: {y_test_best_answer}")

Confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix
cf_matrix = confusion_matrix(y_test_best_answer, y_pred_best_answer)
sns.heatmap(cf_matrix, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names)
plt.show()

Accuracy across categories

In [ ]:
class_correct, class_count = [0]*2, [0]*2

for i in range(y_test.shape[0]):    
    if (y_test_best_answer[i] == y_pred_best_answer[i]):
        class_correct[y_test_best_answer[i]] +=1
    class_count[y_test_best_answer[i]] += 1
    
for i in range(len(class_names)):
    print (f"Accuracy for {class_names[i]}: {class_correct[i]/class_count[i]:.2%}")

### ROC curve
The ROC curve shows the trade-off between sensitivity (True Positive Rate) and the false positive rate (False Positive Rate). An **AUC** close to 1 indicates a good classifier, AUC = 0.5 corresponds to chance.

In [ ]:
from sklearn.metrics import roc_curve, auc

fpr, tpr, _ = roc_curve(y_test_best_answer, y_pred[:, 1])
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"SimpleRNN  (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], 'k--', label='Random classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — SimpleRNN')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

# GRU model
The model will be very similar; we just replace the SimpleRNN part with GRU.

In [ ]:
from keras.layers import GRU
gru_model = Sequential(name="GRU_Model")
gru_model.add(Embedding(vocabulary_size, embd_len))
gru_model.add(GRU(128, activation='tanh', return_sequences=False))
gru_model.add(Dense(2, activation='softmax'))

Displaying the network structure

In [ ]:
gru_model.summary()

### Training the GRU neural network

In [ ]:
gru_model.compile(
    loss="categorical_crossentropy",
    optimizer='adam',
    metrics=['accuracy']
)

In [ ]:
gru_history = gru_model.fit(X_train, y_train,
                         batch_size=64,
                         epochs=15,
                         verbose=1,
                         validation_data=(X_valid, y_valid))

Saving the trained model

In [ ]:
gru_model.save('rnn_gru.keras')

### Displaying the training history

In [ ]:
fig2 = plt.figure()                
plt.plot(gru_history.history['loss'], label='Train Loss')
plt.plot(gru_history.history['accuracy'], label='Train Accuracy')
plt.plot(gru_history.history['val_loss'], label='Validation Loss')
plt.plot(gru_history.history['val_accuracy'], label='Validation Accuracy')
plt.legend(loc="right")
plt.title('Loss, accuracy')
plt.ylabel('Loss, accuracy')
plt.xlabel('Number of epoch')
plt.show()   

### Model verification

In [ ]:
scores = gru_model.evaluate(X_test, y_test)
print (f"Loss function: {scores[0]}")
print (f"Accuracy: {scores[1]}")

Predicting the test data

In [ ]:
y_pred = gru_model.predict(X_test)

Evaluation results for the first review

In [ ]:
print (f"Prediction: {y_pred[0]}")
print (f"Reality: {y_test[0]}")

Predictions and reality

In [ ]:
y_pred_best_answer = np.argmax(y_pred, axis=-1)
y_test_best_answer=np.argmax(y_test, axis=-1)
print (f"Predictions: {y_pred_best_answer}")
print (f"Reality: {y_test_best_answer}")

Confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix
cf_matrix = confusion_matrix(y_test_best_answer, y_pred_best_answer)
sns.heatmap(cf_matrix, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names)
plt.show()

Accuracy across categories

In [ ]:
class_correct, class_count = [0]*2, [0]*2

for i in range(y_test.shape[0]):    
    if (y_test_best_answer[i] == y_pred_best_answer[i]):
        class_correct[y_test_best_answer[i]] +=1
    class_count[y_test_best_answer[i]] += 1
    
for i in range(len(class_names)):
    print (f"Accuracy for {class_names[i]}: {class_correct[i]/class_count[i]:.2%}")

### ROC curve

In [ ]:
from sklearn.metrics import roc_curve, auc

fpr, tpr, _ = roc_curve(y_test_best_answer, y_pred[:, 1])
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"GRU  (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], 'k--', label='Random classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — GRU')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

# LSTM model
Let's try an LSTM model. Again we only swap out that part of the network.

In [ ]:
from keras.layers import LSTM

In [ ]:
lstm_model = Sequential(name="LSTM_Model")
lstm_model.add(Embedding(vocabulary_size, embd_len))
lstm_model.add(LSTM(128, activation='tanh', return_sequences=False))
lstm_model.add(Dense(2, activation='softmax'))

Displaying the network structure

In [ ]:
lstm_model.summary()

### Training the neural network

In [ ]:
lstm_model.compile(
    loss="categorical_crossentropy",
    optimizer='adam',
    metrics=['accuracy']
)

In [ ]:
lstm_history = lstm_model.fit(X_train, y_train,
                          batch_size=64,
                          epochs=15,
                          verbose=1,
                          validation_data=(X_valid, y_valid))

Saving the trained network

In [ ]:
lstm_model.save('rnn_lstm.keras')

### Displaying the training history

In [ ]:
fig3 = plt.figure()                
plt.plot(lstm_history.history['loss'], label='Train Loss')
plt.plot(lstm_history.history['accuracy'], label='Train Accuracy')
plt.plot(lstm_history.history['val_loss'], label='Validation Loss')
plt.plot(lstm_history.history['val_accuracy'], label='Validation Accuracy')
plt.legend(loc="right")
plt.title('Loss, accuracy')
plt.ylabel('Loss, accuracy')
plt.xlabel('Number of epochs')
plt.show()

### Model verification

In [ ]:
scores = lstm_model.evaluate(X_test, y_test)
print (f"Loss function: {scores[0]}")
print (f"Accuracy: {scores[1]}")

Predicting the test data

In [ ]:
y_pred = lstm_model.predict(X_test)

Evaluation results for the first review

In [ ]:
print (f"Prediction: {y_pred[0]}")
print (f"Reality: {y_test[0]}")

Predictions and reality

In [ ]:
y_pred_best_answer = np.argmax(y_pred, axis=-1)
y_test_best_answer=np.argmax(y_test, axis=-1)
print (f"Predictions: {y_pred_best_answer}")
print (f"Reality: {y_test_best_answer}")

Confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix
cf_matrix = confusion_matrix(y_test_best_answer, y_pred_best_answer)
sns.heatmap(cf_matrix, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names)
plt.show()

Accuracy across categories

In [ ]:
class_correct, class_count = [0]*2, [0]*2

for i in range(y_test.shape[0]):    
    if (y_test_best_answer[i] == y_pred_best_answer[i]):
        class_correct[y_test_best_answer[i]] +=1
    class_count[y_test_best_answer[i]] += 1
    
for i in range(len(class_names)):
    print (f"Accuracy for {class_names[i]}: {class_correct[i]/class_count[i]:.2%}")

### ROC curve

In [ ]:
from sklearn.metrics import roc_curve, auc

fpr, tpr, _ = roc_curve(y_test_best_answer, y_pred[:, 1])
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"LSTM  (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], 'k--', label='Random classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — LSTM')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

# Bi-directional LSTM Model
Finally, let's try a bidirectional LSTM model.

In [ ]:
from keras.layers import Bidirectional

In [ ]:
bi_lstm_model = Sequential(name="Bidirectional_LSTM")
bi_lstm_model.add(Embedding(vocabulary_size, embd_len))
bi_lstm_model.add(Bidirectional(LSTM(128, activation='tanh', return_sequences=False)))
bi_lstm_model.add(Dense(2, activation='softmax'))

Printing the network structure

In [ ]:
bi_lstm_model.summary()

### Training the network

In [ ]:
bi_lstm_model.compile(
  loss="categorical_crossentropy",
  optimizer='adam',
  metrics=['accuracy']
)

In [ ]:
bi_lstm_history = bi_lstm_model.fit(X_train, y_train,
                             batch_size=64,
                             epochs=15,
                             validation_data=(X_valid, y_valid))

Saving the trained model

In [ ]:
bi_lstm_model.save('rnn_bi_lstm.keras')

### Displaying the training history

In [ ]:
fig4 = plt.figure()                
plt.plot(bi_lstm_history.history['loss'], label='Train Loss')
plt.plot(bi_lstm_history.history['accuracy'], label='Train Accuracy')
plt.plot(bi_lstm_history.history['val_loss'], label='Validation Loss')
plt.plot(bi_lstm_history.history['val_accuracy'], label='Validation Accuracy')
plt.legend(loc="right")
plt.title('Loss, accuracy')
plt.ylabel('Loss, accuracy')
plt.xlabel('Number of epoch')
plt.show() 

### Model verification

In [ ]:
scores = bi_lstm_model.evaluate(X_test, y_test)
print (f"Loss function: {scores[0]}")
print (f"Accuracy: {scores[1]}")

Predicting the test data

In [ ]:
y_pred = bi_lstm_model.predict(X_test)

Evaluation results for the first review

In [ ]:
print (f"Prediction: {y_pred[0]}")
print (f"Reality: {y_test[0]}")

Predictions and reality

In [ ]:
y_pred_best_answer = np.argmax(y_pred, axis=-1)
y_test_best_answer=np.argmax(y_test, axis=-1)
print (f"Predictions: {y_pred_best_answer}")
print (f"Reality: {y_test_best_answer}")

Confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix
cf_matrix = confusion_matrix(y_test_best_answer, y_pred_best_answer)
sns.heatmap(cf_matrix, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names)
plt.show()

Accuracy across categories

In [ ]:
class_correct, class_count = [0]*2, [0]*2

for i in range(y_test.shape[0]):    
    if (y_test_best_answer[i] == y_pred_best_answer[i]):
        class_correct[y_test_best_answer[i]] +=1
    class_count[y_test_best_answer[i]] += 1
    
for i in range(len(class_names)):
    print (f"Accuracy for {class_names[i]}: {class_correct[i]/class_count[i]:.2%}")

### ROC curve

In [ ]:
from sklearn.metrics import roc_curve, auc

fpr, tpr, _ = roc_curve(y_test_best_answer, y_pred[:, 1])
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"Bidirectional LSTM  (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], 'k--', label='Random classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Bidirectional LSTM')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## Comparison of ROC curves for all models
All four models on a single chart for easy comparison.

In [ ]:
from sklearn.metrics import roc_curve, auc

y_true = np.argmax(y_test, axis=-1)

models = [
    ("SimpleRNN",          RNN_model),
    ("GRU",                gru_model),
    ("LSTM",               lstm_model),
    ("Bidirectional LSTM", bi_lstm_model),
]

fig, ax = plt.subplots(figsize=(8, 6))
for name, m in models:
    pred = m.predict(X_test, verbose=0)
    fpr, tpr, _ = roc_curve(y_true, pred[:, 1])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f"{name}  (AUC = {roc_auc:.3f})")

ax.plot([0, 1], [0, 1], 'k--', label='Random classifier')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — all models')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()